In [ ]:

# ============ IMPORTS ============
import pandas as pd
import numpy as np
import mlflow
from src.calculate_ml_metrics import calculate_ml_metrics
from src.calculate_trade_metrics import calculate_trade_metrics
import quantstats as qs
import matplotlib.pyplot as plt

# =========== CONFIGURATION ============
feature_set = "confirmed" # "all" or "confirmed" -  features confirmed by boruta algorithm
window = 7
model_names = ["baseline_stratified", "baseline_buy_hold", 
               "log_reg", "random_forest", "svc", "xgboost"
]
# ======================================

# ============ LOAD LABELLED DATA ============
labelled_data_cache = pd.read_pickle('data_cache/04_labelled_data.pkl')
returns_train = labelled_data_cache['returns_train']
returns_test = labelled_data_cache['returns_test']

# ============ LABEL REMAPPING ============
label_map   = {-1: 0, 0: 1, 1: 2}
label_unmap = { 0: -1, 1: 0, 2: 1}

In [ ]:
# =========== MLFLOW SETUP ============
experiment  = mlflow.get_experiment_by_name("07_triple_barrier_classification")
experiment_id = experiment.experiment_id

# =========== LIST PREVIOUS RUNS ============
ml_runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id],
                            filter_string = "tags.Hyperpar_optimised_for = 'Balanced_F1_macro'", #tags.Hyperpar_optimised_for = 'GT_Score', status = 'FINISHED
                            order_by = ['start_time DESC'])

# ========== LOAD LATEST MODELS ==========
latest_run_df = ml_runs.groupby('tags.mlflow.runName').first()

In [ ]:
# =========== INITIALISE DICTIONARIES ============
models = {}
train_predictions = {}
test_predictions = {}
selected_features = {}
# =========== LOAD MODELS AND PREDICTIONS ============
for model_name in model_names:
    run_name = f"{model_name}_{feature_set}_features"
    try:
        run_id = latest_run_df.loc[run_name, "run_id"]
        models[model_name] = mlflow.sklearn.load_model(f"runs:/{run_id}/model")

        train_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path=f"07_final_ml_train_data_{model_name}_{feature_set}.pkl"
        )
        test_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path=f"07_final_ml_test_data_{model_name}_{feature_set}.pkl"
        )
        
        train_predictions[model_name] = pd.read_pickle(train_path)
        test_predictions[model_name]  = pd.read_pickle(test_path)

    except KeyError:
        print(f"No finished run found for '{run_name}' — skipping.")

In [ ]:
# =========== EVALUATION ============
ml_metrics = []
for model_name in model_names:
    try:
        train_df = train_predictions[model_name]
        test_df = test_predictions[model_name]
        y_train = train_df[f"Label_{window}day"]
        train_pred = train_df[f"pred_{model_name}"]
        train_proba = train_df[f"proba_{model_name}"]
        y_test = test_df[f"Label_{window}day"]
        test_pred = test_df[f"pred_{model_name}"]
        test_proba = test_df[f"proba_{model_name}"]
        
        train_ml_metrics_dict = calculate_ml_metrics(y_train, train_pred, train_proba, "train")
        test_ml_metrics_dict = calculate_ml_metrics(y_test, test_pred, test_proba, "test")
        
        train_f1 = train_ml_metrics_dict["train_f1_macro"]
        test_f1  = test_ml_metrics_dict["test_f1_macro"]
        overfit_metrics =  {
            "overfit_gap": round(train_f1 - test_f1, 4),
            "gen_ratio":   round(test_f1 / max(train_f1, 1e-6), 4)
        }
        ml_metrics.append({"Model": model_name, **train_ml_metrics_dict, **test_ml_metrics_dict, **overfit_metrics})
    except KeyError:
        print(f"No stored dataframes for '{model_name}' — skipping.")
        
ml_metrics = pd.DataFrame(ml_metrics).set_index("Model").T

In [ ]:
# ============ TRADING METRICS ============
trade_metrics = []
daily_returns = {}
for model_name in model_names:
    try:
        train_df = train_predictions[model_name]
        test_df = test_predictions[model_name]
        y_train = train_df[f"Label_{window}day"]
        train_pred = train_df[f"pred_{model_name}"]
        y_test = test_df[f"Label_{window}day"]
        test_pred = test_df[f"pred_{model_name}"]

        train_trade_metrics_dict, _ = calculate_trade_metrics(y_train, train_pred, returns_train, window, "train")
        test_trade_metrics_dict, test_daily_returns = calculate_trade_metrics(y_test,  test_pred,  returns_test,  window, "test")
        
        trade_metrics.append({"Model": model_name, **train_trade_metrics_dict, **test_trade_metrics_dict})
        daily_returns[model_name] = test_daily_returns
    except KeyError:
        print(f"No stored dataframes for '{model_name}' — skipping.")

# ============ SHOW & STORE RESULTS ============
trade_metrics = pd.DataFrame(trade_metrics).set_index("Model").T

# ============ PASSIVE BUY AND HOLD ============
btc_cumulative   = (1 + returns_test).cumprod()
btc_total_return = btc_cumulative.iloc[-1] - 1  # as a decimal

btc_benchmark = {
    "test_trade_count": int(returns_test.count()),
    "test_trade_frequency_pct": 100.0,
    "test_win_rate": qs.stats.win_rate(returns_test),
    "test_profit_factor": qs.stats.profit_factor(returns_test),
    "test_avg_win_pct": qs.stats.avg_win(returns_test) * 100,
    "test_avg_loss_pct": abs(qs.stats.avg_loss(returns_test)) * 100,
    "test_cum_return_pct": btc_total_return * 100,
    "test_sharpe_ratio": qs.stats.sharpe(returns_test, periods=365),
    "test_sortino_ratio": qs.stats.sortino(returns_test, periods=365),
    "test_max_drawdown": qs.stats.max_drawdown(returns_test),
    "test_calmar_ratio": qs.stats.calmar(returns_test),
}

trade_metrics["btc_passive"] = pd.Series(btc_benchmark)
daily_returns["btc_passive"] = returns_test

In [ ]:
# =========== HELPER FUNCTION ===========
def get_evaluator_config(model_name):    
    base = {
        "log_explainer": True,
        "explainer_type": "auto",
        "explainability_nsamples": 200,
        "explainability_kernel_link": "identity",
        "log_model_explanations": False,
        "max_error_examples": 50,
    }
    if model_name == "log_reg":
        base["explainer_type"] = "linear"
    elif model_name == "svc":
        base["explainability_nsamples"] = 15
    return base

# ============ LOG METRICS + MLFLOW EVALUATE ============
for model_name in model_names:
    run_name = f"{model_name}_{feature_set}_features"
    try:
        run_id   = latest_run_df.loc[run_name, "run_id"]
        test_df  = test_predictions[model_name]
        y_test   = test_df[f"Label_{window}day"]
        test_daily_returns = daily_returns[model_name]

        # Reconstruct X_test (features only)
        feature_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path=f"features/07_final_ml_features_{model_name}_{feature_set}.pkl"
        )
        selected_features[model_name] = pd.read_pickle(feature_path)
        X_test_eval = test_df[selected_features[model_name]]

        with mlflow.start_run(run_id=run_id):
            # Log custom metrics
            for metric, value in ml_metrics[model_name].items():
                mlflow.log_metric(metric, value)
            for metric, value in trade_metrics[model_name].items():
                mlflow.log_metric(metric, value)

            # Log ml and trade metrics dataframes
            ml_metrics.to_csv(f'data_cache/08_final_ml_metrics.csv')
            trade_metrics.to_csv(f'data_cache/08_final_trade_metrics.csv')
            mlflow.log_artifact(f'data_cache/08_final_ml_metrics.csv',artifact_path='evaluation')
            mlflow.log_artifact(f'data_cache/08_final_trade_metrics.csv',artifact_path='evaluation')

            # MLflow evaluate (SHAP plots, confusion matrix etc.)
            if not model_name.startswith("baseline"):
                eval_data = X_test_eval.copy()
                eval_data[f"Label_{window}day"] = y_test.map(label_map) if model_name == "xgboost" else y_test

                mlflow.models.evaluate(
                    model=f"runs:/{run_id}/model",
                    data=eval_data,
                    targets=f"Label_{window}day",
                    model_type="classifier",
                    evaluator_config=get_evaluator_config(model_name)
                )
                # Log Tearsheet
                qs.reports.html(
                    test_daily_returns,
                    output=f"data_cache/08_qs_tearsheet_{model_name}_{feature_set}.html",
                    title=f"{model_name} Trading Performance"
                )
                mlflow.log_artifact(f"data_cache/08_qs_tearsheet_{model_name}_{feature_set}.html", artifact_path="trade_plots")
            
            # Log BTC Passive benchmark
            qs.reports.html(
                daily_returns["btc_passive"],
                output=f"data_cache/08_tearsheet_btc_passive_{feature_set}.html",
                title="BTC Passive Buy-and-Hold Benchmark"
            )

    except KeyError:
        print(f"No finished run found for '{run_name}' — skipping.")